# Thai Election OCR Pipeline

In [ ]:
!pip install pandas tqdm Pillow rapidfuzz google-genai

In [ ]:
import os
import re
import json
import time
import traceback
from io import BytesIO
from typing import List, Dict, Any, Tuple
from collections import defaultdict
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from rapidfuzz import fuzz, process
from google import genai
from google.genai import types


/Users/nonbangkok/Downloads/final_data/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# =========================
# 1) Config
# =========================

# IMAGES_DIR = "images"
IMAGES_DIR = "table_manual_crop"
TEMPLATE_PATH = "submission_template_old.csv"

OUTPUT_PATH = "submission_gemini_strict_total.csv"

DOC_CACHE_PATH = "gemini_doc_cache_strict_total.json"
ROW_CACHE_PATH = "gemini_row_cache_strict_total.json"
PROGRESS_PATH = "gemini_progress_strict_total.json"
DEBUG_ROWS_PATH = "gemini_debug_rows_strict_total.json"

RAW_RESPONSE_DIR = "gemini_raw_responses_strict_total"
BAD_JSON_DIR = "gemini_bad_json_strict_total"

# MODEL_NAME = "gemini-2.5-flash"
MODEL_NAME = "gemini-3.1-flash-lite-preview"

MAX_RETRIES = 6
SLEEP_BASE = 2.0
RATE_LIMIT_WAIT = 10

MAX_IMAGE_SIDE = 2200
JPEG_QUALITY = 88

STRICT_MATCH_THRESHOLD = 92
LOOSE_MATCH_THRESHOLD = 78

TOTAL_VALIDATION_ROUNDS = 3
TOTAL_TOLERANCE_ABS = 0
TOTAL_SOFT_TOLERANCE_ABS = 5
TOTAL_HARD_SKIP_PCT = 0.08

FILL_EMPTY_WITH_ZERO = True

os.makedirs(RAW_RESPONSE_DIR, exist_ok=True)
os.makedirs(BAD_JSON_DIR, exist_ok=True)


In [ ]:
# =========================
# 2) API key
# =========================

# api_key = os.environ.get("GEMINI_API_KEY", "").strip()
# if not api_key:
#     raise ValueError("Please set GEMINI_API_KEY in environment.")

api_key = "GEMINI_API_KEY"
client = genai.Client(api_key=api_key)

In [ ]:
# =========================
# 3) Utility
# =========================

THAI_DIGIT_MAP = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")

THAI_UNITS = {
    "ศูนย์": 0, "หนึ่ง": 1, "เอ็ด": 1, "สอง": 2, "สาม": 3, "สี่": 4,
    "ห้า": 5, "หก": 6, "เจ็ด": 7, "แปด": 8, "เก้า": 9, "ยี่": 2,
}
THAI_MULTIPLIERS = {
    "สิบ": 10, "ร้อย": 100, "พัน": 1000, "หมื่น": 10000, "แสน": 100000, "ล้าน": 1000000,
}

def thai_text_to_number(text: str) -> int:
    text = str(text or "").strip()
    if not text:
        return -1

    vocab = {}
    vocab.update(THAI_UNITS)
    vocab.update(THAI_MULTIPLIERS)

    tokens = []
    pos = 0
    while pos < len(text):
        matched = False
        for length in range(min(6, len(text) - pos), 0, -1):
            piece = text[pos:pos + length]
            if piece in vocab:
                if piece in THAI_UNITS:
                    tokens.append(("unit", THAI_UNITS[piece]))
                else:
                    tokens.append(("mult", THAI_MULTIPLIERS[piece]))
                pos += length
                matched = True
                break
        if not matched:
            pos += 1

    if not tokens:
        return -1

    result = 0
    current = 0
    for typ, val in tokens:
        if typ == "unit":
            current = val
        else:
            if current == 0:
                current = 1
            result += current * val
            current = 0
    result += current
    return result

def save_json(path: str, obj: Any):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def load_json(path: str, default):
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return default

def normalize_whitespace(s: str) -> str:
    return re.sub(r"\s+", " ", str(s or "")).strip()

def normalize_digits(text: str) -> str:
    return str(text or "").translate(THAI_DIGIT_MAP)

def extract_digits_only(text: str) -> str:
    text = str(text or "").strip()
    if not text:
        return ""

    thai_num = thai_text_to_number(text)
    if thai_num >= 0 and re.fullmatch(r"[\u0E00-\u0E7F\s]+", text):
        return str(thai_num)

    text = normalize_digits(text)
    text = re.sub(r"[()]", "", text)
    text = re.sub(r"[, ]", "", text)
    text = re.sub(r"[^0-9]", "", text)
    return text

def normalize_party_name(text: str) -> str:
    text = str(text or "")
    text = normalize_digits(text)
    text = text.replace("พรรคการเมือง", "")
    text = text.replace("ชื่อพรรค", "")
    text = text.replace("ชื่อ พรรค", "")
    text = re.sub(r"\(.*?\)", " ", text)
    text = re.sub(r"[^\w\u0E00-\u0E7F]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text

def parse_doc_id_from_filename(filename: str) -> Tuple[str, int]:
    name = os.path.splitext(os.path.basename(filename))[0]
    m = re.match(r"^(.*?_\d+_\d+)(?:_page(\d+))?$", name)
    if not m:
        raise ValueError(f"Cannot parse filename: {filename}")
    doc_id = m.group(1)
    page_num = int(m.group(2)) if m.group(2) else 1
    return doc_id, page_num

def list_doc_pages(images_dir: str) -> Dict[str, List[Tuple[int, str]]]:
    grouped = defaultdict(list)
    for fn in os.listdir(images_dir):
        if fn.lower().endswith(".png"):
            doc_id, page_num = parse_doc_id_from_filename(fn)
            grouped[doc_id].append((page_num, os.path.join(images_dir, fn)))
    for doc_id in grouped:
        grouped[doc_id] = sorted(grouped[doc_id], key=lambda x: x[0])
    return dict(grouped)

def resize_image(img: Image.Image, max_side: int = MAX_IMAGE_SIDE) -> Image.Image:
    w, h = img.size
    scale = min(max_side / max(w, h), 1.0)
    if scale < 1.0:
        return img.resize((int(w * scale), int(h * scale)))
    return img

def pil_to_jpeg_bytes(img: Image.Image, quality: int = JPEG_QUALITY) -> bytes:
    if img.mode not in ("RGB", "L"):
        img = img.convert("RGB")
    buf = BytesIO()
    img.save(buf, format="JPEG", quality=quality, optimize=True)
    return buf.getvalue()

def build_image_parts(image_pages: List[Tuple[int, str]]) -> List[Any]:
    parts = []
    for _, path in image_pages:
        img = Image.open(path)
        img = resize_image(img)
        img_bytes = pil_to_jpeg_bytes(img)
        parts.append(types.Part.from_bytes(data=img_bytes, mime_type="image/jpeg"))
    return parts

def is_rate_limit_error(err: Exception) -> bool:
    err_str = str(err)
    err_type = type(err).__name__
    return "429" in err_str or "RESOURCE_EXHAUSTED" in err_str or "ResourceExhausted" in err_type

def retry_generate(model: str, contents: List[Any], config: types.GenerateContentConfig):
    last_err = None
    for attempt in range(MAX_RETRIES):
        try:
            return client.models.generate_content(model=model, contents=contents, config=config)
        except Exception as e:
            last_err = e
            if is_rate_limit_error(e):
                wait = RATE_LIMIT_WAIT
                print(f"[Rate-limit Retry {attempt+1}/{MAX_RETRIES}] waiting {wait}s ...")
            else:
                wait = SLEEP_BASE * (2 ** attempt)
                print(f"[Retry {attempt+1}/{MAX_RETRIES}] {type(e).__name__}: {e}")
            if attempt < MAX_RETRIES - 1:
                time.sleep(wait)
    raise last_err

def extract_json_block(text: str) -> str:
    text = text.strip()
    text = re.sub(r"^```json\s*", "", text)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end != -1 and end > start:
        text = text[start:end + 1]
    return text.strip()

def repair_json_text(text: str) -> str:
    text = extract_json_block(text)
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f]", "", text)
    text = text.replace("“", '"').replace("”", '"').replace("‘", "'").replace("’", "'")
    text = re.sub(r"\bTrue\b", "true", text)
    text = re.sub(r"\bFalse\b", "false", text)
    text = re.sub(r"\bNone\b", "null", text)
    text = re.sub(r",\s*([}\]])", r"\1", text)
    return text

def safe_json_loads(text: str, save_prefix: str):
    raw = text or ""
    fixed = repair_json_text(raw)
    try:
        return json.loads(fixed)
    except json.JSONDecodeError:
        with open(os.path.join(BAD_JSON_DIR, f"{save_prefix}_raw.txt"), "w", encoding="utf-8") as f:
            f.write(raw)
        with open(os.path.join(BAD_JSON_DIR, f"{save_prefix}_fixed.txt"), "w", encoding="utf-8") as f:
            f.write(fixed)
        raise

def save_raw_response(name: str, text: str):
    with open(os.path.join(RAW_RESPONSE_DIR, f"{name}.txt"), "w", encoding="utf-8") as f:
        f.write(text or "")


In [ ]:
# =========================
# 4) Template
# =========================

template_df = pd.read_csv(TEMPLATE_PATH)
required_cols = {"id", "doc_id", "row_num", "party_name"}
missing = required_cols - set(template_df.columns)
if missing:
    raise ValueError(f"TEMPLATE missing columns: {missing}")

template_df["party_name"] = template_df["party_name"].fillna("").astype(str)
template_df["party_name_norm"] = template_df["party_name"].map(normalize_party_name)

doc_to_template_rows = {
    doc_id: g.sort_values("row_num").copy()
    for doc_id, g in template_df.groupby("doc_id", sort=False)
}


In [ ]:
# =========================
# 5) Prompts
# =========================

SYSTEM_PROMPT = """
You are extracting structured vote table data from scanned Thai election result forms.

Rules:
1. Read the table visually from the provided page image(s).
2. Return ONLY valid JSON.
3. Convert Thai numerals (๐๑๒๓๔๕๖๗๘๙) to Arabic digits (0123456789).
4. Convert Thai text numbers to Arabic digits before output.
5. If a vote number appears inside parentheses, output only the number inside the parentheses.
6. votes must contain digits only. No commas, spaces, text, parentheses.
7. If a row's vote count cannot be read, set votes to "".
8. Preserve party names as seen on the page as best as possible.
9. For constituency documents, each row may include candidate_name, party_name, votes.
10. For party_list documents, each row includes party_name and votes.
11. Do not hallucinate rows that are not on the page(s).
12. Ignore headers, footnotes, signatures, stamps, page numbers.
13. Only set has_grand_total_row=true if you can clearly see the final summary row such as:
    - "รวมคะแนนทั้งสิ้น"
    - "รวมคะแนนทั้งหมด"
    - "รวมคะแนน"
14. If that row is not clearly visible, set:
    - "has_grand_total_row": false
    - "total_votes": ""
15. Never guess total_votes from a normal candidate row or a normal party row.
16. Use double quotes for all JSON keys and string values.
17. No markdown fences, no comments, no trailing commas.
18. Preserve row order from top to bottom, left to right.
"""

def build_single_page_prompt(doc_id: str, doc_type: str, page_num: int) -> str:
    if doc_type == "party_list":
        return f"""
Document ID: {doc_id}
Document type: party_list
Page: {page_num}

Extract ONLY the vote-table rows visible on this page.

Important:
- Set has_grand_total_row=true ONLY if the page clearly contains the final summary row
  like "รวมคะแนนทั้งสิ้น" or equivalent.
- If not clearly visible, set has_grand_total_row=false and total_votes="".

Return JSON exactly like this:
{{
  "doc_id": "{doc_id}",
  "doc_type": "party_list",
  "page": {page_num},
  "has_grand_total_row": false,
  "total_votes": "",
  "rows": [
    {{
      "page": {page_num},
      "row_order_on_page": 1,
      "party_name": "...",
      "votes": "12345"
    }}
  ]
}}
"""
    return f"""
Document ID: {doc_id}
Document type: constituency
Page: {page_num}

Extract ONLY the candidate rows visible on this page.

Important:
- Set has_grand_total_row=true ONLY if the page clearly contains the final summary row
  like "รวมคะแนนทั้งสิ้น" or equivalent.
- If not clearly visible, set has_grand_total_row=false and total_votes="".

Return JSON exactly like this:
{{
  "doc_id": "{doc_id}",
  "doc_type": "constituency",
  "page": {page_num},
  "has_grand_total_row": false,
  "total_votes": "",
  "rows": [
    {{
      "page": {page_num},
      "row_order_on_page": 1,
      "candidate_name": "...",
      "party_name": "...",
      "votes": "12345"
    }}
  ]
}}
"""

def build_multi_page_rescue_prompt(
    doc_id: str,
    doc_type: str,
    page_nums: List[int],
    expected_total: int,
    previous_rows: List[Dict[str, Any]],
    template_count: int
) -> str:
    prev_summary = json.dumps(
        [
            {
                "page": r.get("page"),
                "row_order_on_page": r.get("row_order_on_page"),
                "party_name": r.get("party_name", ""),
                "candidate_name": r.get("candidate_name", ""),
                "votes": r.get("votes", "")
            }
            for r in previous_rows
        ],
        ensure_ascii=False,
        indent=2
    )

    if doc_type == "party_list":
        return f"""
Document ID: {doc_id}
Document type: party_list
Pages provided: {page_nums}

IMPORTANT:
- The official grand total on the form is {expected_total}.
- The document should have exactly {template_count} party rows in total.
- Read ALL provided pages together as ONE document.
- Keep rows in reading order (page ascending, top to bottom).
- Re-check every digit carefully.
- Set total_votes to {expected_total} only if the grand-total row is actually visible in the provided pages.

Previous extraction:
{prev_summary}

Return JSON exactly like this:
{{
  "doc_id": "{doc_id}",
  "doc_type": "party_list",
  "has_grand_total_row": true,
  "total_votes": "{expected_total}",
  "rows": [
    {{
      "page": 2,
      "row_order_on_page": 1,
      "party_name": "...",
      "votes": "12345"
    }}
  ]
}}
"""
    return f"""
Document ID: {doc_id}
Document type: constituency
Pages provided: {page_nums}

IMPORTANT:
- The official grand total on the form is {expected_total}.
- Read ALL provided pages together as ONE document.
- Keep rows in reading order (page ascending, top to bottom).
- Re-check every digit carefully.
- Set total_votes to {expected_total} only if the grand-total row is actually visible in the provided pages.

Previous extraction:
{prev_summary}

Return JSON exactly like this:
{{
  "doc_id": "{doc_id}",
  "doc_type": "constituency",
  "has_grand_total_row": true,
  "total_votes": "{expected_total}",
  "rows": [
    {{
      "page": 1,
      "row_order_on_page": 1,
      "candidate_name": "...",
      "party_name": "...",
      "votes": "12345"
    }}
  ]
}}
"""


In [ ]:
# =========================
# 6) Cleaning helpers
# =========================

def clean_rows(rows: List[Dict[str, Any]], doc_type: str, default_page: int = 1) -> List[Dict[str, Any]]:
    cleaned = []
    for i, r in enumerate(rows, start=1):
        if not isinstance(r, dict):
            continue
        row = dict(r)
        row["page"] = int(row.get("page", default_page) or default_page)
        row["row_order_on_page"] = int(row.get("row_order_on_page", i) or i)
        row["party_name"] = normalize_whitespace(row.get("party_name", ""))
        row["votes"] = extract_digits_only(row.get("votes", ""))
        if doc_type == "constituency":
            row["candidate_name"] = normalize_whitespace(row.get("candidate_name", ""))
        cleaned.append(row)
    cleaned.sort(key=lambda x: (int(x.get("page", 0)), int(x.get("row_order_on_page", 0))))
    return cleaned

def compute_row_sum(rows: List[Dict[str, Any]]) -> int:
    total = 0
    for r in rows:
        v = r.get("votes", "")
        if v:
            try:
                total += int(v)
            except ValueError:
                pass
    return total

def validate_total_votes(rows: List[Dict[str, Any]], total_votes_str: str, allow_soft: bool = False) -> bool:
    if not total_votes_str:
        return True

    try:
        expected_total = int(total_votes_str)
    except ValueError:
        return True

    row_sum = compute_row_sum(rows)
    diff = row_sum - expected_total
    abs_diff = abs(diff)

    if abs_diff <= TOTAL_TOLERANCE_ABS:
        return True

    if allow_soft and abs_diff <= TOTAL_SOFT_TOLERANCE_ABS:
        print(f"  [TOTAL SOFT-CLOSE] sum={row_sum}, total={expected_total}, diff={diff}")
        return True

    print(f"  [TOTAL MISMATCH] sum={row_sum}, expected={expected_total}, diff={diff}")
    return False

def rows_by_page(rows: List[Dict[str, Any]]) -> Dict[int, List[Dict[str, Any]]]:
    out = defaultdict(list)
    for r in rows:
        out[int(r.get("page", 1))].append(r)
    for p in out:
        out[p] = sorted(out[p], key=lambda x: int(x.get("row_order_on_page", 0)))
    return dict(out)

def rebuild_all_rows(page_rows_map: Dict[int, List[Dict[str, Any]]]) -> List[Dict[str, Any]]:
    all_rows = []
    for page_num in sorted(page_rows_map.keys()):
        all_rows.extend(sorted(page_rows_map[page_num], key=lambda x: int(x.get("row_order_on_page", 0))))
    return all_rows

def page_debug_stats(page_rows_map: Dict[int, List[Dict[str, Any]]]) -> Dict[int, Dict[str, int]]:
    stats = {}
    for page_num, rows in sorted(page_rows_map.items()):
        stats[page_num] = {
            "rows": len(rows),
            "blank_votes": sum(1 for r in rows if not r.get("votes")),
            "sum": compute_row_sum(rows),
        }
    return stats


In [ ]:
# =========================
# 7) Gemini extraction
# =========================

def call_gemini_json(parts: List[Any], save_prefix: str, temperature: float = 0.0) -> Dict[str, Any]:
    contents = [types.Content(role="user", parts=parts)]
    config = types.GenerateContentConfig(
        temperature=temperature,
        top_p=0.95,
        top_k=32,
        max_output_tokens=16384,
        response_mime_type="application/json",
    )

    response = retry_generate(MODEL_NAME, contents, config)
    raw_text = response.text or ""
    save_raw_response(save_prefix, raw_text)

    try:
        return safe_json_loads(raw_text, save_prefix=save_prefix)
    except Exception:
        repair_prompt = """
Return ONLY valid JSON.
Use double quotes for every key and every string value.
No markdown fences.
No comments.
No trailing commas.
"""
        retry_parts = [types.Part.from_text(text=repair_prompt)] + parts
        response = retry_generate(MODEL_NAME, [types.Content(role="user", parts=retry_parts)], config)
        raw_text = response.text or ""
        save_raw_response(save_prefix + "_retry", raw_text)
        return safe_json_loads(raw_text, save_prefix=save_prefix + "_retry")

def extract_single_page_with_gemini(doc_id: str, page_num: int, image_path: str) -> Tuple[List[Dict[str, Any]], str, bool]:
    doc_type = "party_list" if doc_id.startswith("party_list_") else "constituency"
    prompt = build_single_page_prompt(doc_id, doc_type, page_num)
    parts = [
        types.Part.from_text(text=SYSTEM_PROMPT),
        types.Part.from_text(text=prompt),
        *build_image_parts([(page_num, image_path)])
    ]
    data = call_gemini_json(parts, save_prefix=f"{doc_id}_page{page_num}", temperature=0.0)
    rows = clean_rows(data.get("rows", []), doc_type, default_page=page_num)

    has_total = bool(data.get("has_grand_total_row", False))
    total_votes = extract_digits_only(data.get("total_votes", "")) if has_total else ""

    return rows, total_votes, has_total

def extract_multi_page_rescue(
    doc_id: str,
    image_pages: List[Tuple[int, str]],
    expected_total: int,
    previous_rows: List[Dict[str, Any]]
) -> Tuple[List[Dict[str, Any]], str]:
    doc_type = "party_list" if doc_id.startswith("party_list_") else "constituency"
    page_nums = [p for p, _ in image_pages]
    template_count = len(doc_to_template_rows.get(doc_id, []))
    prompt = build_multi_page_rescue_prompt(doc_id, doc_type, page_nums, expected_total, previous_rows, template_count)
    parts = [
        types.Part.from_text(text=SYSTEM_PROMPT),
        types.Part.from_text(text=prompt),
        *build_image_parts(image_pages)
    ]
    data = call_gemini_json(parts, save_prefix=f"{doc_id}_doc_rescue", temperature=0.1)
    rows = clean_rows(data.get("rows", []), doc_type, default_page=page_nums[0] if page_nums else 1)

    has_total = bool(data.get("has_grand_total_row", False))
    total_votes = extract_digits_only(data.get("total_votes", "")) if has_total else str(expected_total)

    return rows, total_votes

def targeted_recheck_page(
    doc_id: str,
    page_num: int,
    image_path: str,
    previous_rows: List[Dict[str, Any]],
    expected_total: int,
    current_sum: int
) -> Tuple[List[Dict[str, Any]], str]:
    doc_type = "party_list" if doc_id.startswith("party_list_") else "constituency"
    prev_summary = json.dumps(
        [
            {
                "row_order_on_page": r.get("row_order_on_page"),
                "party_name": r.get("party_name", ""),
                "candidate_name": r.get("candidate_name", ""),
                "votes": r.get("votes", "")
            }
            for r in previous_rows
        ],
        ensure_ascii=False,
        indent=2
    )

    if doc_type == "party_list":
        prompt = f"""
Document ID: {doc_id}
Document type: party_list
Page: {page_num}

IMPORTANT:
- Official grand total on the full document is {expected_total}.
- Current full-document extracted sum is {current_sum}.
- Re-read ONLY this page.
- Keep row order from top to bottom.
- Re-check every digit carefully, especially similar Thai digits.
- Return only rows visible on this page.
- If the grand-total row is not clearly visible on this page, total_votes must be "".

Previous extraction for this page:
{prev_summary}

Return JSON exactly like this:
{{
  "doc_id": "{doc_id}",
  "doc_type": "party_list",
  "page": {page_num},
  "has_grand_total_row": false,
  "total_votes": "",
  "rows": [
    {{
      "page": {page_num},
      "row_order_on_page": 1,
      "party_name": "...",
      "votes": "12345"
    }}
  ]
}}
"""
    else:
        prompt = f"""
Document ID: {doc_id}
Document type: constituency
Page: {page_num}

IMPORTANT:
- Official grand total on the full document is {expected_total}.
- Current full-document extracted sum is {current_sum}.
- Re-read ONLY this page.
- Keep row order from top to bottom.
- Re-check every digit carefully, especially similar Thai digits.
- Return only rows visible on this page.
- If the grand-total row is not clearly visible on this page, total_votes must be "".

Previous extraction for this page:
{prev_summary}

Return JSON exactly like this:
{{
  "doc_id": "{doc_id}",
  "doc_type": "constituency",
  "page": {page_num},
  "has_grand_total_row": false,
  "total_votes": "",
  "rows": [
    {{
      "page": {page_num},
      "row_order_on_page": 1,
      "candidate_name": "...",
      "party_name": "...",
      "votes": "12345"
    }}
  ]
}}
"""

    parts = [
        types.Part.from_text(text=SYSTEM_PROMPT),
        types.Part.from_text(text=prompt),
        *build_image_parts([(page_num, image_path)])
    ]
    try:
        data = call_gemini_json(parts, save_prefix=f"{doc_id}_page{page_num}_recheck", temperature=0.2)
        rows = clean_rows(data.get("rows", []), doc_type, default_page=page_num)
        total_votes = extract_digits_only(data.get("total_votes", "")) if bool(data.get("has_grand_total_row", False)) else ""
        if not rows:
            return previous_rows, total_votes
        return rows, total_votes
    except Exception as e:
        print(f"  [RECHECK ERROR] {doc_id} page {page_num}: {type(e).__name__}: {e}")
        return previous_rows, ""

def page_priority_order(page_rows_map: Dict[int, List[Dict[str, Any]]], image_pages: List[Tuple[int, str]]) -> List[int]:
    max_page = max(p for p, _ in image_pages)
    scored = []
    for page_num, rows in page_rows_map.items():
        blank_votes = sum(1 for r in rows if not r.get("votes"))
        short_votes = sum(1 for r in rows if r.get("votes") and len(r.get("votes")) <= 2)
        score = blank_votes * 10 + short_votes * 2 + (5 if page_num == max_page else 0)
        scored.append((score, page_num))
    scored.sort(reverse=True)
    order = [p for _, p in scored]
    if not order:
        order = [p for p, _ in image_pages]
    return order

def extract_doc_rows_with_gemini(doc_id: str, image_pages: List[Tuple[int, str]]) -> Dict[str, Any]:
    doc_type = "party_list" if doc_id.startswith("party_list_") else "constituency"
    page_rows_map = {}
    doc_total_votes = ""

    last_page_num = max(p for p, _ in image_pages)

    # --- Initial page-by-page pass ---
    for page_num, image_path in image_pages:
        try:
            rows, page_total, has_total = extract_single_page_with_gemini(doc_id, page_num, image_path)
            page_rows_map[page_num] = rows

            accepted_total = ""
            if has_total and page_num == last_page_num and page_total:
                accepted_total = page_total
                doc_total_votes = page_total

            print(
                f"  page {page_num}: extracted {len(rows)} rows"
                + (f", raw_total={page_total}, accepted_total={accepted_total}" if page_total else "")
            )
        except Exception as e:
            print(f"  [PAGE ERROR] {doc_id} page {page_num}: {type(e).__name__}: {e}")
            page_rows_map[page_num] = []

    all_rows = rebuild_all_rows(page_rows_map)
    if doc_total_votes:
        print(f"  page sums: {page_debug_stats(page_rows_map)}")

    # --- Total validation / repair ---
    if doc_total_votes and not validate_total_votes(all_rows, doc_total_votes):
        try:
            expected_total = int(doc_total_votes)
        except ValueError:
            expected_total = None

        if expected_total is not None:
            current_diff = abs(compute_row_sum(all_rows) - expected_total)
            reference = max(compute_row_sum(all_rows), expected_total, 1)

            if current_diff > reference * TOTAL_HARD_SKIP_PCT:
                print(f"  [TOTAL SKIP] diff={current_diff} > {int(reference * TOTAL_HARD_SKIP_PCT)} ; total or rows likely too noisy")
            else:
                # 1) Rescue with multi-page reread
                try:
                    rescue_rows, rescue_total = extract_multi_page_rescue(doc_id, image_pages, expected_total, all_rows)
                    rescue_diff = abs(compute_row_sum(rescue_rows) - expected_total)
                    print(f"  [DOC RESCUE] diff {current_diff} -> {rescue_diff}")
                    if rescue_rows and rescue_diff < current_diff:
                        all_rows = rescue_rows
                        current_diff = rescue_diff
                        if rescue_total:
                            doc_total_votes = rescue_total
                        page_rows_map = rows_by_page(all_rows)
                except Exception as e:
                    print(f"  [DOC RESCUE ERROR] {doc_id}: {type(e).__name__}: {e}")

                # 2) Page replacement: reread one page at a time
                for round_idx in range(1, TOTAL_VALIDATION_ROUNDS + 1):
                    if validate_total_votes(all_rows, doc_total_votes):
                        break

                    improved_this_round = False
                    current_sum = compute_row_sum(all_rows)
                    order = page_priority_order(page_rows_map, image_pages)
                    print(f"  [TOTAL REPAIR ROUND {round_idx}/{TOTAL_VALIDATION_ROUNDS}] page order={order}, sum={current_sum}, expected={expected_total}")

                    page_path_map = {p: path for p, path in image_pages}

                    for page_num in order:
                        prev_rows = page_rows_map.get(page_num, [])
                        new_rows, page_total = targeted_recheck_page(
                            doc_id=doc_id,
                            page_num=page_num,
                            image_path=page_path_map[page_num],
                            previous_rows=prev_rows,
                            expected_total=expected_total,
                            current_sum=compute_row_sum(all_rows),
                        )

                        candidate_map = dict(page_rows_map)
                        candidate_map[page_num] = new_rows
                        candidate_all_rows = rebuild_all_rows(candidate_map)
                        candidate_diff = abs(compute_row_sum(candidate_all_rows) - expected_total)

                        if candidate_diff < current_diff:
                            print(f"    [ACCEPT PAGE {page_num}] diff {current_diff} -> {candidate_diff}")
                            page_rows_map = candidate_map
                            all_rows = candidate_all_rows
                            current_diff = candidate_diff
                            improved_this_round = True
                            if page_total and page_num == last_page_num:
                                doc_total_votes = page_total
                            if validate_total_votes(all_rows, doc_total_votes):
                                break
                        else:
                            print(f"    [REJECT PAGE {page_num}] diff would be {current_diff} -> {candidate_diff}")

                    if not improved_this_round:
                        print("  [STOP] no page improved global diff")
                        break

                if not validate_total_votes(all_rows, doc_total_votes):
                    print(f"  [TOTAL WARN] {doc_id}: still mismatched after repair; final diff={abs(compute_row_sum(all_rows) - int(doc_total_votes))}")

    return {
        "doc_id": doc_id,
        "doc_type": doc_type,
        "total_votes": doc_total_votes,
        "rows": all_rows,
    }


In [ ]:
# =========================
# 8) Matching
# =========================

def best_match_row(extracted_party: str, candidates_norm: List[str]) -> Tuple[int, int]:
    ep = normalize_party_name(extracted_party)
    if not ep:
        return -1, 0
    best = process.extractOne(ep, candidates_norm, scorer=fuzz.WRatio)
    if best is None:
        return -1, 0
    _, score, idx = best
    return idx, int(score)

def map_party_list_by_order_if_possible(doc_id: str, extracted_rows: List[Dict[str, Any]]) -> Dict[int, str]:
    template_rows = doc_to_template_rows[doc_id].copy()
    row_nums = template_rows["row_num"].tolist()

    ordered_rows = sorted(extracted_rows, key=lambda x: (int(x.get("page", 0)), int(x.get("row_order_on_page", 0))))
    if len(ordered_rows) != len(row_nums):
        return {}

    output = {}
    for row_num, row in zip(row_nums, ordered_rows):
        votes = extract_digits_only(row.get("votes", ""))
        if votes == "":
            return {}
        output[row_num] = votes
    return output

def map_extracted_rows_to_template(doc_id: str, extracted_doc: Dict[str, Any]) -> Dict[int, str]:
    template_rows = doc_to_template_rows[doc_id].copy()
    template_norms = template_rows["party_name_norm"].tolist()
    row_nums = template_rows["row_num"].tolist()
    candidates = extracted_doc.get("rows", [])

    if doc_id.startswith("party_list_"):
        direct_map = map_party_list_by_order_if_possible(doc_id, candidates)
        if direct_map:
            return direct_map

    output = {}
    used_template_indices = set()
    scored_matches = []

    for row in candidates:
        party_name = row.get("party_name", "")
        votes = extract_digits_only(row.get("votes", ""))
        if not party_name or votes == "":
            continue
        idx, score = best_match_row(party_name, template_norms)
        if idx >= 0:
            scored_matches.append((score, idx, row))

    scored_matches.sort(reverse=True, key=lambda x: x[0])

    for score, idx, row in scored_matches:
        if idx in used_template_indices:
            continue
        if score >= STRICT_MATCH_THRESHOLD:
            used_template_indices.add(idx)
            output[row_nums[idx]] = extract_digits_only(row.get("votes", ""))

    remaining_template = [i for i in range(len(template_norms)) if i not in used_template_indices]
    for row in candidates:
        party_name = row.get("party_name", "")
        votes = extract_digits_only(row.get("votes", ""))
        if not party_name or votes == "":
            continue

        ep = normalize_party_name(party_name)
        best_idx = -1
        best_score = -1
        for i in remaining_template:
            score = fuzz.WRatio(ep, template_norms[i])
            if score > best_score:
                best_score = score
                best_idx = i

        if best_idx >= 0 and best_score >= LOOSE_MATCH_THRESHOLD:
            row_num = row_nums[best_idx]
            if row_num not in output:
                output[row_num] = votes
                used_template_indices.add(best_idx)
                remaining_template.remove(best_idx)

    return output


In [ ]:
# =========================
# 9) Main pipeline
# =========================

doc_cache = load_json(DOC_CACHE_PATH, {})
row_cache = load_json(ROW_CACHE_PATH, {})
progress = load_json(PROGRESS_PATH, {"done_docs": [], "errors": {}})
debug_rows = load_json(DEBUG_ROWS_PATH, {})

done_docs = set(progress.get("done_docs", []))
errors = progress.get("errors", {})

doc_pages = list_doc_pages(IMAGES_DIR)
all_doc_ids = sorted(doc_pages.keys())

print("Total docs found:", len(all_doc_ids))
print("Already done:", len(done_docs))

for doc_id in tqdm(all_doc_ids):
    if doc_id in done_docs:
        continue

    try:
        if doc_id not in doc_to_template_rows:
            print(f"[SKIP] {doc_id} not found in template")
            continue

        pages = doc_pages[doc_id]

        print("\n" + "=" * 90)
        print(f"[DOC] {doc_id}")
        print("pages:", [p for p, _ in pages])

        extracted_doc = extract_doc_rows_with_gemini(doc_id, pages)
        doc_cache[doc_id] = extracted_doc
        debug_rows[doc_id] = extracted_doc.get("rows", [])

        print(f"Extracted rows total: {len(extracted_doc.get('rows', []))}")

        row_map = map_extracted_rows_to_template(doc_id, extracted_doc)
        row_cache[doc_id] = {str(k): v for k, v in row_map.items()}

        print(f"Matched rows: {len(row_map)} / {len(doc_to_template_rows[doc_id])}")

        done_docs.add(doc_id)
        errors.pop(doc_id, None)

        save_json(DOC_CACHE_PATH, doc_cache)
        save_json(ROW_CACHE_PATH, row_cache)
        save_json(DEBUG_ROWS_PATH, debug_rows)
        save_json(PROGRESS_PATH, {
            "done_docs": sorted(done_docs),
            "errors": errors,
        })

    except Exception as e:
        errors[doc_id] = {
            "error_type": type(e).__name__,
            "error": str(e),
            "traceback": traceback.format_exc()[-4000:],
        }
        save_json(PROGRESS_PATH, {
            "done_docs": sorted(done_docs),
            "errors": errors,
        })
        print(f"\n[ERROR] {doc_id}: {type(e).__name__}: {e}\n")

print("\n" + "=" * 90)
print("Done docs:", len(done_docs))
print("Errors:", len(errors))


Total docs found: 300
Already done: 300


100%|██████████| 300/300 [00:00<00:00, 3061535.77it/s]


Done docs: 300
Errors: 0


In [ ]:
# =========================
# 10) Reprocess failed/empty docs from first run and update caches/progress/debug json
# =========================

# ---- pick target doc_ids: empty debug rows or in errors ----
empty_debug_docs = [doc_id for doc_id, rows in debug_rows.items() if not rows]
error_docs = list(errors.keys())

target_doc_ids = sorted(set(empty_debug_docs + error_docs))

print("Reprocess targets:", target_doc_ids)
print("Total:", len(target_doc_ids))

# ---- reprocess ----
for doc_id in target_doc_ids:
    if doc_id not in doc_pages:
        print(f"[SKIP] {doc_id} not found in images")
        continue
    if doc_id not in doc_to_template_rows:
        print(f"[SKIP] {doc_id} not found in template")
        continue

    try:
        print("\n" + "=" * 90)
        print(f"[RERUN] {doc_id}")
        pages = doc_pages[doc_id]

        extracted_doc = extract_doc_rows_with_gemini(doc_id, pages)
        doc_cache[doc_id] = extracted_doc
        debug_rows[doc_id] = extracted_doc.get("rows", [])

        row_map = map_extracted_rows_to_template(doc_id, extracted_doc)
        row_cache[doc_id] = {str(k): v for k, v in row_map.items()}

        done_docs.add(doc_id)
        errors.pop(doc_id, None)

        save_json(DOC_CACHE_PATH, doc_cache)
        save_json(ROW_CACHE_PATH, row_cache)
        save_json(DEBUG_ROWS_PATH, debug_rows)
        save_json(PROGRESS_PATH, {
            "done_docs": sorted(done_docs),
            "errors": errors,
        })

        print(f"[OK] rows={len(extracted_doc.get('rows', []))}, matched={len(row_map)}")

    except Exception as e:
        errors[doc_id] = {
            "error_type": type(e).__name__,
            "error": str(e),
            "traceback": traceback.format_exc()[-4000:],
        }
        save_json(PROGRESS_PATH, {
            "done_docs": sorted(done_docs),
            "errors": errors,
        })
        print(f"[ERROR] {doc_id}: {type(e).__name__}: {e}")

print("\nReprocess done.")
print("Done docs:", len(done_docs))
print("Errors:", len(errors))

Reprocess targets: ['constituency_10_1', 'constituency_10_10', 'constituency_10_2', 'constituency_10_20', 'constituency_10_31', 'constituency_10_32', 'constituency_10_33', 'party_list_30_9', 'party_list_31_1', 'party_list_31_10']
Total: 10

[RERUN] constituency_10_1
  page 2: extracted 18 rows, raw_total=77775, accepted_total=77775
  page sums: {2: {'rows': 18, 'blank_votes': 0, 'sum': 76675}}
  [TOTAL MISMATCH] sum=76675, expected=77775, diff=-1100
  [DOC RESCUE] diff 1100 -> 1100
  [TOTAL MISMATCH] sum=76675, expected=77775, diff=-1100
  [TOTAL REPAIR ROUND 1/3] page order=[2], sum=76675, expected=77775
    [ACCEPT PAGE 2] diff 1100 -> 700
  [TOTAL MISMATCH] sum=77075, expected=77775, diff=-700
  [TOTAL MISMATCH] sum=77075, expected=77775, diff=-700
  [TOTAL REPAIR ROUND 2/3] page order=[2], sum=77075, expected=77775
    [REJECT PAGE 2] diff would be 700 -> 700
  [STOP] no page improved global diff
  [TOTAL MISMATCH] sum=77075, expected=77775, diff=-700
  [TOTAL WARN] constituency_10

In [ ]:
# =========================
# 11) Validate + Match report
# =========================

print("\n" + "=" * 90)
print("VALIDATE + MATCH REPORT")
print("=" * 90)

total_template_rows = 0
total_matched_rows = 0
total_extracted_rows = 0
docs_fully_matched = 0
docs_with_total_ok = 0
docs_with_total_close = 0
docs_with_total_mismatch = 0
docs_with_total_skip = 0
docs_without_total = 0

for doc_id in all_doc_ids:
    if doc_id not in doc_to_template_rows:
        continue

    template_count = len(doc_to_template_rows[doc_id])
    extracted_count = len(doc_cache.get(doc_id, {}).get("rows", []))
    matched_count = len(row_cache.get(doc_id, {}))

    total_template_rows += template_count
    total_extracted_rows += extracted_count
    total_matched_rows += matched_count

    if matched_count == template_count:
        docs_fully_matched += 1

    doc_total = doc_cache.get(doc_id, {}).get("total_votes", "")
    if doc_total:
        row_sum = compute_row_sum(doc_cache.get(doc_id, {}).get("rows", []))
        try:
            expected = int(doc_total)
            diff = abs(row_sum - expected)
            reference = max(row_sum, expected, 1)

            if diff == 0:
                docs_with_total_ok += 1
            elif diff <= TOTAL_SOFT_TOLERANCE_ABS:
                docs_with_total_close += 1
            elif diff > reference * TOTAL_HARD_SKIP_PCT:
                docs_with_total_skip += 1
            else:
                docs_with_total_mismatch += 1
        except ValueError:
            docs_without_total += 1
    else:
        docs_without_total += 1

match_pct = (total_matched_rows / total_template_rows * 100) if total_template_rows else 0
print(f"Documents processed : {len(done_docs)}")
print(f"Documents with errors: {len(errors)}")
print(f"Template rows total : {total_template_rows}")
print(f"Extracted rows total: {total_extracted_rows}")
print(f"Matched rows total  : {total_matched_rows}")
print(f"Match coverage      : {match_pct:.1f}%")
print(f"Fully matched docs  : {docs_fully_matched} / {len(all_doc_ids)}")
print(f"Total exact match   : {docs_with_total_ok}")
print(f"Total soft-close    : {docs_with_total_close}")
print(f"Total mismatch      : {docs_with_total_mismatch}")
print(f"Total skipped       : {docs_with_total_skip}")
print(f"No total found      : {docs_without_total}")



VALIDATE + MATCH REPORT
Documents processed : 300
Documents with errors: 0
Template rows total : 10053
Extracted rows total: 9973
Matched rows total  : 9965
Match coverage      : 99.1%
Fully matched docs  : 286 / 300
Total exact match   : 116
Total soft-close    : 7
Total mismatch      : 149
Total skipped       : 21
No total found      : 7


In [ ]:
# =========================
# 12) Build submission
# =========================

pred_votes = []
for _, row in template_df.iterrows():
    doc_id = row["doc_id"]
    row_num = int(row["row_num"])
    vote = ""
    if doc_id in row_cache:
        vote = row_cache[doc_id].get(str(row_num), "")
    vote = extract_digits_only(vote)
    pred_votes.append(vote)

submission_df = template_df[["id"]].copy()
submission_df["votes"] = pred_votes

if FILL_EMPTY_WITH_ZERO:
    submission_df["votes"] = submission_df["votes"].replace("", "0")

submission_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved submission to: {OUTPUT_PATH}")

print("\n--- SUMMARY ---")
print("total rows      :", len(submission_df))
print("empty votes     :", (submission_df["votes"] == "").sum())
print("zero votes      :", (submission_df["votes"] == "0").sum())
print(submission_df.head(20).to_string(index=False))

Saved submission to: submission_gemini_strict_total.csv

--- SUMMARY ---
total rows      : 10053
empty votes     : 0
zero votes      : 90
                  id votes
 constituency_10_1_1 14813
 constituency_10_1_2 14368
 constituency_10_1_3   979
 constituency_10_1_4   244
 constituency_10_1_5   351
 constituency_10_1_6 34167
 constituency_10_1_7  6030
 constituency_10_1_8  1023
 constituency_10_1_9  2075
constituency_10_1_10   168
constituency_10_1_11   629
constituency_10_1_12  1133
constituency_10_1_13   113
constituency_10_1_14    94
constituency_10_1_15   165
constituency_10_1_16   489
constituency_10_1_17   154
constituency_10_1_18    80
constituency_10_10_1 41804
constituency_10_10_2 19457
